# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) library. We will:
- Load Croissant metadata and records
- Explore available record sets and fields (by `@id`)
- Load the records into pandas DataFrames by record set
- Apply basic Exploratory Data Analysis (EDA) and process selected numeric fields
- Visualize and summarize the data

### Dataset Source
The dataset is described and documented using a Croissant JSON-LD schema at the URL specified below.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print overview information
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets from the metadata
record_sets = dataset.metadata.record_sets

print("Available Record Sets (name and @id):")
for record_set in record_sets:
    print(f"- {record_set.name} (@id: {record_set.id})")

# Explore fields for each record set
for record_set in record_sets:
    print(f"\nFields for Record Set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"  - {field.name} (@id: {field.id}) type={field.data_type if hasattr(field, 'data_type') else 'N/A'}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data from each record set
dataframes = {}

# We'll store record_set @id as variable for convenient reference
record_set_ids = [r.id for r in record_sets]
for record_set in record_sets:
    # Note that mlcroissant expects @id for the record set
    print(f"Loading records for {record_set.name} (@id: {record_set.id}) ...")
    records_iter = dataset.records(record_set=record_set.id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set.id] = df
    print(f"  Shape: {df.shape}")
    if len(df.columns) > 0:
        print(f"  Columns: {df.columns[:5].tolist()} ...\n")
if not dataframes:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's work with the main protein abundance record set and demonstrate common EDA operations.

The main tabular record set typically contains protein measurements. 

- Replace the variable `main_table_id` below with the `@id` for the main table (see overview above).
- Similarly, choose relevant numeric fields (e.g., 'Molecular Weight', 'Abundance') and a grouping field (e.g., sample name or run).
- Fields MUST be referenced by their `@id`, not display name!

In [ ]:
# Example: Choose the main record set (replace with correct @id)
# For demonstration we pick the first record set. Adjust as needed for your case.
if len(record_set_ids) == 0:
    raise ValueError('No record sets present in metadata!')
main_table_id = record_set_ids[0]
main_df = dataframes[main_table_id]

print(f"Main record set columns (@id): {main_df.columns.tolist()}")

# Set numeric field and group field by their @id (from Section 2 output)
# For example, typical protein datasets include fields like 'cr:MolecularWeight' or 'cr:Abundance'
numeric_field_id = None
group_field_id = None

# Simple heuristic to auto-pick a numeric field
for cname in main_df.columns:
    # Pick a field containing 'weight', 'coverage', 'abundance', or 'peptide' as a demo (you should check actual field ids above)
    if any(s in cname.lower() for s in ['weight', 'coverage', 'abundance', 'peptide', 'mw']):
        numeric_field_id = cname
        break

# Pick a group field for demonstration (e.g., 'cr:Sample' or similar)
for cname in main_df.columns:
    if any(s in cname.lower() for s in ['sample', 'run', 'batch']):
        group_field_id = cname
        break

if numeric_field_id is None:
    print("No numeric field found. Please set 'numeric_field_id' explicitly using the field @id from above.")
if group_field_id is None:
    print("No group field found. You may set 'group_field_id' to a categorical field @id if grouping makes sense.")

# Let's proceed if we have a candidate numeric field
if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Try converting field to numeric (silently ignore errors)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

    # Remove outliers (e.g., negative, zero, or NaN values)
    filtered_df = main_df[main_df[numeric_field_id] > 0].copy()

    # Optionally: filter on a percentile threshold (top 90%)
    threshold = filtered_df[numeric_field_id].quantile(0.1)
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (10th percentile): {len(filtered_df)} records")

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
        print(f"Aggregated stats by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found to analyze.")

## 5. Visualization
Visualize distributions and relationships of selected numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group (if suitable)
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=60)
        plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion
In this notebook, you have:
- Explored the Croissant FAIR^2 mass spectrometry dataset
- Identified record sets and fields by their `@id`
- Extracted and analyzed tabular data using `mlcroissant`
- Filtered, normalized, grouped, and visualized key numeric protein data fields

Further analysis, such as machine learning, dimensionality reduction, or advanced biomedical interpretation, can build upon these cleaned and explored data tables.